# 01 — Data Preparation
### RetailX Demand Forecasting & Inventory Optimization Platform
**CRISP-DM Phase 3 — Data Preparation**

Input: raw Rossmann Store Sales files (`data/raw/`)
Output: a single, clean, merged, analysis-ready dataset (`data/processed/`)

This notebook does **not** perform EDA, feature engineering, forecasting, or modeling —
only the transformations required to produce a trustworthy analytical dataset.

**Dataset note:** this project originally targeted the Kaggle M5 Forecasting – Accuracy dataset,
but switched to Rossmann Store Sales because M5's wide-to-long reshape produces tens of millions
of rows, which is impractical on the local development hardware (8 GB RAM, integrated GPU).
Rossmann is already in long (tidy) format — one row per store per day — so no reshape is needed.

## 0. Environment Setup

**Objective:** Configure paths, logging, and imports so every later cell is reproducible and traceable.

**Business explanation:** A pipeline whose steps aren't logged is not auditable — if a downstream
number looks wrong, an analyst needs to trace it back to a specific transformation and timestamp.

**Implementation:** Resolve the project root via `pathlib`, add `src/` to the import path, and
initialize a file+console logger that every subsequent section reuses.

In [1]:
from __future__ import annotations

import sys
from pathlib import Path

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR.parent) not in sys.path:
    sys.path.insert(0, str(SRC_DIR.parent))

from src.utils.logging_utils import get_logger
from src.utils.io_utils import load_csv, save_parquet
from src.utils.profiling_utils import profile_dataframe, duplicate_report, detect_outliers_iqr, memory_usage_mb
from src.utils.dtype_utils import downcast_numeric, to_category

RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
LOG_DIR = PROJECT_ROOT / "logs"

logger = get_logger("data_preparation", LOG_DIR)

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 140)

logger.info("Environment ready. PROJECT_ROOT=%s", PROJECT_ROOT)

2026-07-22 09:37:52 | INFO     | data_preparation | Environment ready. PROJECT_ROOT=C:\Users\hp\OneDrive\Desktop\Code\Retail Demand Forecasting & Inventory Optimization Platform


## 1. Load Data

**Objective:** Load the Rossmann source files into memory as DataFrames.

**Business context:** `train.csv` describes *what* each store sold and under *what conditions*
(open, promo, holiday) on each day; `store.csv` describes *where* — the format, assortment, and
competitive/promotional context of each of the 1,115 stores. Together they are the foundation
every later phase (SQL reporting, dashboard, forecasting) is built on.

**How the files relate:**
- `train.csv` — one row per `(Store, Date)`, already long format, with daily `Sales` (the
  forecasting target), `Customers`, `Open`, `Promo`, `StateHoliday`, `SchoolHoliday`.
- `store.csv` — one row per `Store` (1,115 total), giving `StoreType`, `Assortment`,
  competition distance/open-date, and `Promo2` recurring-campaign metadata.
- `test.csv` — same grain as `train.csv` for the ~6-week forecast horizon, without `Sales`/`Customers`.
- `sample_submission.csv` — Kaggle competition artifact only; not used for analytics, loaded for completeness.

**Expected dimensions:** train ≈ 1,017,209 rows × 9 columns; store ≈ 1,115 rows × 10 columns;
test ≈ 41,088 rows × 8 columns.

In [2]:
train = load_csv(RAW_DIR / "train.csv", dtype={"StateHoliday": str}, low_memory=False)
store = load_csv(RAW_DIR / "store.csv")
test = load_csv(RAW_DIR / "test.csv", dtype={"StateHoliday": str}, low_memory=False)
sample_submission = load_csv(RAW_DIR / "sample_submission.csv")

print("train:              ", train.shape)
print("store:              ", store.shape)
print("test:               ", test.shape)
print("sample_submission:  ", sample_submission.shape)

train:               (1017209, 9)
store:               (1115, 10)
test:                (41088, 8)
sample_submission:   (41088, 2)


**Key observations:** *(fill in after running — expect train to have 1,017,209 rows across
1,115 stores spanning 2013-01-01 to 2015-07-31; store to have exactly 1,115 rows, one per store;
test to cover the ~6-week window immediately after the train period.)*

## 2. Initial Profiling

**Objective:** Understand shape, dtypes, memory footprint, and business meaning of every
important column before making any changes.

**Business meaning of key columns:**
- `Store` — links every table together; primary key of `store.csv`.
- `Date`, `DayOfWeek` — the time axis; drives weekly/seasonal demand patterns.
- `Sales` — the forecasting target (daily revenue); `Customers` is a strong training-time
  diagnostic (not available in `test.csv`).
- `Open`, `Promo`, `StateHoliday`, `SchoolHoliday` — the operational/promotional context that
  explains *why* demand moves day to day.
- `StoreType`, `Assortment`, `CompetitionDistance`, `Promo2*` — store-level structural attributes
  that explain *why* one store's baseline demand differs from another's.

**Technical reasoning:** Profiling before cleaning prevents "fixing" things that aren't actually
broken (e.g. missing `Promo2Since*` fields are structurally expected wherever `Promo2 == 0`, not
missing data).

In [3]:
for name, df in [("train", train), ("store", store), ("test", test)]:
    print(f"\n{'='*20} {name} {'='*20}")
    print("Shape:", df.shape)
    print("Memory (deep):", f"{memory_usage_mb(df):.2f} MB")
    print("Duplicate rows:", df.duplicated().sum())
    display(profile_dataframe(df, name).head(15))


==================== train ====================
Shape: (1017209, 9)
Memory (deep): 84.43 MB


Duplicate rows: 0


,dtype,null_count,null_pct,n_unique,memory_mb
Store,int64,0,0.0,1115,8.138
DayOfWeek,int64,0,0.0,7,8.138
Date,str,0,0.0,942,18.310
Sales,int64,0,0.0,21734,8.138
Customers,int64,0,0.0,4086,8.138
Open,int64,0,0.0,2,8.138
Promo,int64,0,0.0,2,8.138
StateHoliday,str,0,0.0,4,9.155
SchoolHoliday,int64,0,0.0,2,8.138



==================== store ====================
Shape: (1115, 10)
Memory (deep): 0.10 MB
Duplicate rows: 0


,dtype,null_count,null_pct,n_unique,memory_mb
Store,int64,0,0.00,1115,0.009
StoreType,str,0,0.00,4,0.010
Assortment,str,0,0.00,3,0.010
CompetitionDistance,float64,3,0.27,654,0.009
CompetitionOpenSinceMonth,float64,354,31.75,12,0.009
CompetitionOpenSinceYear,float64,354,31.75,23,0.009
Promo2,int64,0,0.00,2,0.009
Promo2SinceWeek,float64,544,48.79,24,0.009
Promo2SinceYear,float64,544,48.79,7,0.009
PromoInterval,str,544,48.79,3,0.018



==================== test ====================
Shape: (41088, 8)
Memory (deep): 3.08 MB
Duplicate rows: 0


,dtype,null_count,null_pct,n_unique,memory_mb
Id,int64,0,0.00,41088,0.329
Store,int64,0,0.00,856,0.329
DayOfWeek,int64,0,0.00,7,0.329
Date,str,0,0.00,48,0.740
Open,float64,11,0.03,2,0.329
Promo,int64,0,0.00,2,0.329
StateHoliday,str,0,0.00,2,0.370
SchoolHoliday,int64,0,0.00,2,0.329


In [4]:
train.describe(include="all").T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
Store,1017209.0,NaN,NaN,NaN,558.429727,321.908651,1.0,280.0,558.0,838.0,1115.0
DayOfWeek,1017209.0,NaN,NaN,NaN,3.998341,1.997391,1.0,2.0,4.0,6.0,7.0
Date,1017209,942,2015-07-31,1115,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Sales,1017209.0,NaN,NaN,NaN,5773.818972,3849.926175,0.0,3727.0,5744.0,7856.0,41551.0
Customers,1017209.0,NaN,NaN,NaN,633.145946,464.411734,0.0,405.0,609.0,837.0,7388.0
Open,1017209.0,NaN,NaN,NaN,0.830107,0.375539,0.0,1.0,1.0,1.0,1.0
Promo,1017209.0,NaN,NaN,NaN,0.381515,0.485759,0.0,0.0,0.0,1.0,1.0
StateHoliday,1017209,4,0,986159,NaN,NaN,NaN,NaN,NaN,NaN,NaN
SchoolHoliday,1017209.0,NaN,NaN,NaN,0.178647,0.383056,0.0,0.0,0.0,0.0,1.0


In [5]:
store.describe(include="all").T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
Store,1115.0,NaN,NaN,NaN,558.0,322.01708,1.0,279.5,558.0,836.5,1115.0
StoreType,1115,4,a,602,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Assortment,1115,3,a,593,NaN,NaN,NaN,NaN,NaN,NaN,NaN
CompetitionDistance,1112.0,NaN,NaN,NaN,5404.901079,7663.17472,20.0,717.5,2325.0,6882.5,75860.0
CompetitionOpenSinceMonth,761.0,NaN,NaN,NaN,7.224704,3.212348,1.0,4.0,8.0,10.0,12.0
CompetitionOpenSinceYear,761.0,NaN,NaN,NaN,2008.668857,6.195983,1900.0,2006.0,2010.0,2013.0,2015.0
Promo2,1115.0,NaN,NaN,NaN,0.512108,0.500078,0.0,0.0,1.0,1.0,1.0
Promo2SinceWeek,571.0,NaN,NaN,NaN,23.595447,14.141984,1.0,13.0,22.0,37.0,50.0
Promo2SinceYear,571.0,NaN,NaN,NaN,2011.763573,1.674935,2009.0,2011.0,2012.0,2013.0,2015.0
PromoInterval,571,3,"Jan,Apr,Jul,Oct",335,NaN,NaN,NaN,NaN,NaN,NaN,NaN


**Key observations:** *(fill in after running)* — note which `store.csv` columns carry
expected structural NaNs (`CompetitionDistance`/`CompetitionOpenSince*` when a store has no
tracked competitor; `Promo2Since*`/`PromoInterval` whenever `Promo2 == 0`) vs. columns where a
null would represent a real data-quality problem (`Sales`, `Store`, `StoreType`).

## 3. Data Quality Assessment

**Objective:** Systematically evaluate completeness, consistency, validity, uniqueness, and
integrity across `train`, `store`, and `test`, and *detect* (not treat) outliers.

**Business importance:** Feeding a forecasting model negative sales, duplicated store-day
records, or inconsistent store keys silently produces wrong reorder-point recommendations —
exactly the stockout/overstock problem RetailX is trying to solve.

In [6]:
quality_report = {}

# Completeness
quality_report["train_null_total"] = int(train.isnull().sum().sum())
quality_report["store_null_total"] = int(store.isnull().sum().sum())
quality_report["test_null_total"] = int(test.isnull().sum().sum())

# Uniqueness / duplicates
quality_report["train_dupe_keys"] = duplicate_report(train, ["Store", "Date"])
quality_report["store_dupe_keys"] = duplicate_report(store, ["Store"])
quality_report["test_dupe_keys"] = duplicate_report(test, ["Store", "Date"])

# Validity: negative sales, sales on closed days, closed days with nonzero sales
quality_report["negative_sales_rows"] = int((train["Sales"] < 0).sum())
quality_report["closed_with_sales"] = int(((train["Open"] == 0) & (train["Sales"] > 0)).sum())
quality_report["open_flagged_zero_sales"] = int(((train["Open"] == 1) & (train["Sales"] == 0)).sum())
quality_report["negative_competition_distance"] = int((store["CompetitionDistance"] < 0).sum())

# Referential integrity: every Store in train/test must exist in store.csv
quality_report["train_stores_missing_from_store_csv"] = int(
    (~train["Store"].isin(store["Store"])).sum()
)
quality_report["test_stores_missing_from_store_csv"] = int(
    (~test["Store"].isin(store["Store"])).sum()
)

quality_report

{'train_null_total': 0,
 'store_null_total': 2343,
 'test_null_total': 11,
 'train_dupe_keys': {'full_row_duplicates': 0, 'key_duplicates': 0},
 'store_dupe_keys': {'full_row_duplicates': 0, 'key_duplicates': 0},
 'test_dupe_keys': {'full_row_duplicates': 0, 'key_duplicates': 0},
 'negative_sales_rows': 0,
 'closed_with_sales': 0,
 'open_flagged_zero_sales': 54,
 'negative_competition_distance': 0,
 'train_stores_missing_from_store_csv': 0,
 'test_stores_missing_from_store_csv': 0}

In [7]:
# Outlier detection only (IQR fence on daily Sales, open days only), no treatment/analysis here
open_sales = train.loc[train["Open"] == 1, "Sales"]
outlier_mask = detect_outliers_iqr(open_sales)
print(f"Store-days flagged as outliers by daily-Sales IQR fence: {outlier_mask.sum()} / {len(outlier_mask)}")

Store-days flagged as outliers by daily-Sales IQR fence: 30769 / 844392


**Findings (fill in after running):**
- **Completeness** — `train`/`test` should show zero unexpected nulls; `store` nulls are
  structural (see Section 2) and quantified precisely in Section 5.
- **Duplicates** — composite-key duplicates on `(Store, Date)` / `Store` should be 0; any
  non-zero count is a data-integrity bug.
- **Validity** — negative sales, "closed but sold something," and negative competition distance
  should all be 0; a non-zero `open_flagged_zero_sales` count is expected and legitimate (a store
  can be open with zero sales, e.g. a very slow day) and is not itself an error.
- **Referential integrity** — every `Store` in `train`/`test` must resolve to a row in
  `store.csv`; a non-zero count here would break every downstream join.
- **Outliers** — flagged only; addressed later in EDA/feature engineering, not here.

## 4. Cleaning Strategy

**Objective:** Decide, in writing, how each detected issue class will be handled before touching data.

| Issue | Why it happens | Common retail-industry fix | Our approach | Alternatives | Tradeoff |
|---|---|---|---|---|---|
| `StateHoliday` mixed types (`"0"` string vs. numeric `0`) | Kaggle CSV export inconsistency | Normalize to one categorical dtype | Cast entire column to `str`, keep `"0"` as the "no holiday" label | Coerce to boolean is-holiday flag | Keeping the original a/b/c labels preserves holiday-*type* information (public vs. Easter vs. Christmas), which a boolean flag would destroy |
| `CompetitionDistance` missing (3 stores) | No competitor tracked near that store | Treat as "no nearby competition" | Leave `NaN`; add `has_competition` flag downstream in feature engineering, do not impute a fabricated distance | Impute with max observed distance | Imputing implies a specific competitor exists — false signal to a demand model |
| `CompetitionOpenSinceMonth/Year` missing (354 stores) | Competitor open-date unknown or untracked | Leave NaN, compute "months since competition" only where known | Keep NaN; downstream feature engineering treats missing as "unknown duration," not zero | Impute with 0 (implies competitor opened at store's tracking start) | Zero would fabricate a competition-duration signal that doesn't exist |
| `Promo2Since*`, `PromoInterval` missing (544 stores) | Store never opted into Promo2 (`Promo2 == 0`) | Leave NaN; the absence is fully explained by `Promo2` | Keep NaN, verified 1:1 against `Promo2 == 0` | Fill with sentinel values | The missingness is not random — it is a direct, verifiable consequence of another column, so no fix is needed at all |
| Closed-store zero-sales rows (`Open == 0`) | Store did not trade that day | Retain rows, exclude from demand-level modeling later | Keep rows (needed for open/closed pattern analysis), flag via `Open` | Drop closed-day rows entirely | Dropping would lose legitimate information about closure patterns (holidays, refurbishment) useful for the dashboard and open/closed feature engineering |
| Oversized default dtypes (`int64`, `object` ids) | `pd.read_csv` defaults are conservative | Downcast + categorical encoding | `to_numeric(downcast=...)`, `astype("category")` | Leave as-is | Downstream merges/groupbys are materially slower and heavier in memory otherwise |

**Explicitly not treated here:** true statistical outliers in daily sales — flagged for
awareness, but promotion spikes and holiday effects are legitimate business signal, not data
errors, and removing them would bias a demand model. That judgment belongs in EDA/feature
engineering (later phases), not blind removal here.

## 5. Missing Value Treatment

**Objective:** Apply the decisions from Section 4 — treat only what is safe to treat, and
explicitly explain what is deliberately *not* imputed.

**Why some missing values should NOT be imputed:** a missing `CompetitionDistance` means no
competitor is tracked near that store — inventing a distance would fabricate a competitor that
doesn't exist. Similarly, missing `Promo2Since*` fields are not really "missing" at all; they are
a structural consequence of `Promo2 == 0`. Forcing a value into either would inject false signal,
not fill a genuine gap.

In [8]:
store_clean = store.copy()

# StateHoliday: normalize mixed-type column to a single string category
train["StateHoliday"] = train["StateHoliday"].astype(str).str.strip()
test["StateHoliday"] = test["StateHoliday"].astype(str).str.strip()

# Verify Promo2Since*/PromoInterval missingness is fully explained by Promo2 == 0
promo2_inconsistent = store_clean.loc[
    (store_clean["Promo2"] == 1) & store_clean["Promo2SinceWeek"].isnull()
]
assert len(promo2_inconsistent) == 0, "Found Promo2==1 stores with missing Promo2SinceWeek"

# CompetitionDistance / CompetitionOpenSince* / Promo2Since* / PromoInterval are deliberately
# left as NaN here — each NaN has a well-defined business meaning ("no competitor tracked" /
# "not enrolled in Promo2") that downstream feature engineering encodes explicitly rather than
# this notebook overwriting it with a fabricated value.
print("store.csv null counts after StateHoliday normalization elsewhere:")
print(store_clean.isnull().sum())

store.csv null counts after StateHoliday normalization elsewhere:
Store                          0
StoreType                      0
Assortment                     0
CompetitionDistance            3
CompetitionOpenSinceMonth    354
CompetitionOpenSinceYear     354
Promo2                         0
Promo2SinceWeek              544
Promo2SinceYear              544
PromoInterval                544
dtype: int64


**Key observations:** `StateHoliday` now carries a single consistent string dtype across
`train`/`test`; the `Promo2` ↔ `Promo2Since*` consistency assertion passing confirms the
missingness in `store.csv` is structural, not a data-quality defect — so Section 5 makes zero
speculative imputations, only the type-normalization every downstream step depends on.

## 6. Duplicate Handling

**Objective:** Define and enforce composite primary keys, then verify no duplicate demand
records exist.

**Primary / composite keys:**
- `train` / `test`: `(Store, Date)` — one row per store per day.
- `store`: `Store` — one row per physical store.

**How duplicate detection works:** a duplicate on the *full row* only catches exact copies; a
duplicate on the *composite key* catches conflicting records (e.g. two different sales totals
for the same store/day), which is the more dangerous case.

**Why duplicate sales records are dangerous:** a duplicated demand row silently doubles apparent
sales for that store/day. Fed into a reorder-point calculation, that directly causes
over-ordering — the exact overstocking problem RetailX is trying to eliminate.

In [9]:
dupe_checks = {
    "train": duplicate_report(train, ["Store", "Date"]),
    "store": duplicate_report(store_clean, ["Store"]),
    "test": duplicate_report(test, ["Store", "Date"]),
}
dupe_checks

{'train': {'full_row_duplicates': 0, 'key_duplicates': 0},
 'store': {'full_row_duplicates': 0, 'key_duplicates': 0},
 'test': {'full_row_duplicates': 0, 'key_duplicates': 0}}

In [10]:
train = train.drop_duplicates(subset=["Store", "Date"], keep="last")
store_clean = store_clean.drop_duplicates(subset=["Store"], keep="last")
test = test.drop_duplicates(subset=["Store", "Date"], keep="last")

print("Post-dedup shapes:", train.shape, store_clean.shape, test.shape)

Post-dedup shapes: (1017209, 9) (1115, 10) (41088, 8)


**Key observations:** *(fill in after running)* — expect zero composite-key duplicates in
the Rossmann release; the `drop_duplicates` calls are defensive/idempotent guardrails for future
data refreshes, not a fix for a problem found in this specific extract.

## 7. Data Type Optimization

**Objective:** Convert columns to the smallest correct dtype to cut memory footprint and speed
up joins/groupbys in every later phase.

**Memory optimization reasoning:** `object` dtype strings (`StateHoliday`, `StoreType`,
`Assortment`, ...) are stored as Python object pointers — expensive to hash and compare.
Converting low-cardinality identifiers to `category` stores each value once and uses integer
codes internally. Downcasting `int64`/`float64` to the smallest safe width (`int8`/`int16`,
`float32`) further shrinks memory with no loss of information for this data's actual value
ranges.

In [11]:
mem_before = {
    "train": memory_usage_mb(train),
    "store": memory_usage_mb(store_clean),
    "test": memory_usage_mb(test),
}
mem_before

{'train': 84.555631, 'store': 0.100373, 'test': 3.086868}

In [12]:
train["Date"] = pd.to_datetime(train["Date"])
test["Date"] = pd.to_datetime(test["Date"])

train = to_category(train, ["StateHoliday"])
train = downcast_numeric(
    train,
    int_cols=["Store", "DayOfWeek", "Sales", "Customers", "Open", "Promo", "SchoolHoliday"],
    float_cols=[],
)

test = to_category(test, ["StateHoliday"])
test = downcast_numeric(test, int_cols=["Store", "DayOfWeek", "Promo", "SchoolHoliday"], float_cols=[])
# Open has a small number of NaNs by Kaggle convention in test.csv (assumed open) - keep float-safe
test["Open"] = test["Open"].fillna(1)
test = downcast_numeric(test, int_cols=["Open"], float_cols=[])

store_clean = to_category(store_clean, ["StoreType", "Assortment", "PromoInterval"])
store_clean = downcast_numeric(
    store_clean,
    int_cols=["Store", "Promo2"],
    float_cols=["CompetitionDistance", "CompetitionOpenSinceMonth", "CompetitionOpenSinceYear",
                "Promo2SinceWeek", "Promo2SinceYear"],
)

In [13]:
mem_after = {
    "train": memory_usage_mb(train),
    "store": memory_usage_mb(store_clean),
    "test": memory_usage_mb(test),
}
comparison = pd.DataFrame({"before_mb": mem_before, "after_mb": mem_after})
comparison["reduction_pct"] = ((comparison["before_mb"] - comparison["after_mb"]) / comparison["before_mb"] * 100).round(1)
comparison

,before_mb,after_mb,reduction_pct
train,84.555631,21.361558,74.7
store,0.100373,0.029258,70.9
test,3.086868,0.945175,69.4


**Key observations:** *(fill in after running)* — expect the largest relative reduction on
`train` (over a million `int64` rows compressing to `int8`/`int16`/`category`); `test.csv`'s
Kaggle-documented convention of a small number of missing `Open` values is resolved by assuming
the store trades (`Open = 1`) unless stated otherwise, matching the competition's official
guidance.

## 8. Merge Data

**Objective:** Attach store attributes (format, assortment, competition, Promo2 metadata) to
every daily sales record.

**Join key:** `Store` — a single many-to-one join from the daily fact tables to the store
dimension table. Unlike M5 (which needed a wide-to-long melt plus two separate merges against a
calendar table and a weekly price table), Rossmann's `train.csv`/`test.csv` are already long-format
with the date embedded directly, so a single merge is sufficient.

**Join type:** a **left join** anchored on `train`/`test`, because every sales record must be
preserved — sales is the fact table and `store` is a dimension table. An inner join risks
silently dropping real sales history if a store attribute row is unexpectedly missing.

**Potential merge issues:** fan-out (a many-to-many key match inflating row count) and silent row
loss from a wrong join type. Both are checked explicitly below.

In [14]:
row_count_before_merge = len(train)

merged = train.merge(store_clean, on="Store", how="left", validate="many_to_one")
assert len(merged) == row_count_before_merge, "Row count changed after store merge (fan-out or drop)"
assert merged["StoreType"].isnull().sum() == 0, "Some Store IDs in train did not match store.csv"

print("Final merged shape:", merged.shape)
merged.head()

Final merged shape: (1017209, 18)


,Store,DayOfWeek,Date,Sales,Customers,Open,Promo,StateHoliday,SchoolHoliday,StoreType,Assortment,CompetitionDistance,CompetitionOpenSinceMonth,CompetitionOpenSinceYear,Promo2,Promo2SinceWeek,Promo2SinceYear,PromoInterval
0,1,5,2015-07-31,5263,555,1,1,0,1,c,a,1270.0,9.0,2008.0,0,NaN,NaN,NaN
1,2,5,2015-07-31,6064,625,1,1,0,1,a,a,570.0,11.0,2007.0,1,13.0,2010.0,"Jan,Apr,Jul,Oct"
2,3,5,2015-07-31,8314,821,1,1,0,1,a,a,14130.0,12.0,2006.0,1,14.0,2011.0,"Jan,Apr,Jul,Oct"
3,4,5,2015-07-31,13995,1498,1,1,0,1,c,c,620.0,9.0,2009.0,0,NaN,NaN,NaN
4,5,5,2015-07-31,4822,559,1,1,0,1,a,a,29910.0,4.0,2015.0,0,NaN,NaN,NaN


**Key observations:** `validate="many_to_one"` fails loudly on any fan-out, so a passing run
is itself proof the join key behaved as designed; a zero `StoreType` null count after the merge
confirms full referential integrity between `train` and `store.csv`.

## 9. Post-Merge Validation

**Objective:** Confirm the merged dataset is internally consistent before handing it off as the
Phase 3 deliverable.

In [15]:
validation = {}

validation["row_count_matches_expected"] = len(merged) == row_count_before_merge
validation["no_full_row_duplicates"] = merged.duplicated().sum() == 0
validation["date_range"] = (merged["Date"].min(), merged["Date"].max())
validation["n_unique_dates"] = merged["Date"].nunique()
validation["n_unique_stores"] = merged["Store"].nunique()
validation["sales_null_count"] = int(merged["Sales"].isnull().sum())
validation["negative_sales_count"] = int((merged["Sales"] < 0).sum())
validation["closed_with_sales_count"] = int(((merged["Open"] == 0) & (merged["Sales"] > 0)).sum())
validation["competition_distance_missing_pct"] = round(merged["CompetitionDistance"].isnull().mean() * 100, 2)

validation

{'row_count_matches_expected': True,
 'no_full_row_duplicates': np.True_,
 'date_range': (Timestamp('2013-01-01 00:00:00'),
  Timestamp('2015-07-31 00:00:00')),
 'n_unique_dates': 942,
 'n_unique_stores': 1115,
 'sales_null_count': 0,
 'negative_sales_count': 0,
 'closed_with_sales_count': 0,
 'competition_distance_missing_pct': np.float64(0.26)}

In [16]:
assert validation["row_count_matches_expected"], "Row count no longer matches pre-merge train"
assert validation["no_full_row_duplicates"], "Full-row duplicates present in final dataset"
assert validation["sales_null_count"] == 0, "Sales column must never be null"
assert validation["negative_sales_count"] == 0, "Negative sales units are invalid"
assert validation["closed_with_sales_count"] == 0, "A closed store cannot record sales"
print("All post-merge validation checks passed.")

All post-merge validation checks passed.


**Key observations:** *(fill in after running)* — `competition_distance_missing_pct`
reflects genuine "no tracked competitor" stores, not a data-quality defect; a nonzero
`sales_null_count` or any failed assert would mean the dataset is not safe to hand off and
Sections 4–8 need revisiting.

## Save Processed Dataset

Persist the validated, merged dataset as Parquet (preserves dtypes, compresses well, loads fast)
for use by EDA, SQL reporting, the dashboard, and feature engineering in later phases. The
prepared `test.csv` (with store attributes attached) is saved alongside it for the forecasting
phase's inference step.

In [17]:
output_path = PROCESSED_DIR / "rossmann_train_store_merged.parquet"
save_parquet(merged, output_path)
print("Saved processed training dataset to:", output_path)

test_merged = test.merge(store_clean, on="Store", how="left", validate="many_to_one")
assert test_merged["StoreType"].isnull().sum() == 0, "Some Store IDs in test did not match store.csv"
test_output_path = PROCESSED_DIR / "rossmann_test_store_merged.parquet"
save_parquet(test_merged, test_output_path)
print("Saved processed test/forecast-horizon dataset to:", test_output_path)

Saved processed training dataset to: C:\Users\hp\OneDrive\Desktop\Code\Retail Demand Forecasting & Inventory Optimization Platform\data\processed\rossmann_train_store_merged.parquet
Saved processed test/forecast-horizon dataset to: C:\Users\hp\OneDrive\Desktop\Code\Retail Demand Forecasting & Inventory Optimization Platform\data\processed\rossmann_test_store_merged.parquet


## Summary

| Metric | Value |
|---|---|
| Final grain | one row per (Store, Date) |
| Source files | 2 (train, store) merged into 1 training dataset; test merged separately |
| Transformation | direct left join on `Store` (no reshape needed — already long format) |
| Validation | row-count, duplicate, null, and closed-store-sales assertions all pass before save |

**Next phase (not started here):** Phase 4 — Feature Engineering (see
[`docs/Phase4_Feature_Engineering_Strategy.md`](../docs/Phase4_Feature_Engineering_Strategy.md)).